# 05 – Generate GitHub Dashboard

Denne notebooken genererer et statisk GitHub Pages-dashboard fra Gold-data i Databricks.

**Notebooken eier ingen HTML, CSS, JavaScript eller SQL direkte.**

Den leser fra:

- `templates/index.html.tpl` — HTML-template med `$`-plassholdere
- `sql/github_dashboard/*.sql` — SQL-spørringer med `$catalog` / `$schema`
- `assets/css/*.css` — stilark (valideres, kopieres ikke)
- `assets/js/charts.js`, `table.js`, `main.js` — frontend-logikk (valideres)

Den genererer kun:

- `assets/js/data.js` — dashboarddata som `window.dashboardData = { ... }`
- `index.html` — ferdig HTML fra template + KPI-verdier

## Imports og konfigurasjon

In [ ]:
import json
from datetime import datetime
from pathlib import Path
from string import Template
from typing import Any
import pandas as pd

In [0]:
CATALOG = "pensjon_lakehouse"
SCHEMA = "gold"

# Sett denne manuelt hvis automatisk repo-detektering ikke fungerer.
# Eksempel:
# PROJECT_ROOT_OVERRIDE = "/Workspace/Repos/bruker@example.com/Pensjon-Lakehouse"
PROJECT_ROOT_OVERRIDE = None

GENERATED_DATE = datetime.now().strftime("%Y-%m-%d %H:%M")

print(f"Generator startet: {GENERATED_DATE}")

## Prosjektstier

In [0]:
def resolve_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()

    # Databricks: hent notebook-path fra kontekst
    try:
        notebook_path = (
            dbutils
            .notebook
            .entry_point
            .getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )

        workspace_path = Path("/Workspace" + notebook_path)

        if workspace_path.parent.name == "notebooks":
            return workspace_path.parent.parent

        for candidate in [workspace_path.parent, *workspace_path.parents]:
            if candidate.name == "Pensjon-Lakehouse":
                return candidate
    except Exception:
        pass

    # Fallback: working directory
    cwd = Path.cwd().resolve()

    if cwd.name == "notebooks":
        return cwd.parent

    for candidate in [cwd, *cwd.parents]:
        if (candidate / "README.md").exists() and (candidate / "notebooks").exists():
            return candidate
        if candidate.name == "Pensjon-Lakehouse":
            return candidate

    return cwd


In [ ]:
PROJECT_ROOT = resolve_project_root()

# Alt som hører til den statiske GitHub Pages-showcasen ligger her.
SHOWCASE_DIR  = PROJECT_ROOT / "showcase-page"
TEMPLATE_PATH = SHOWCASE_DIR / "templates" / "index.html.tpl"
SQL_DIR       = SHOWCASE_DIR / "sql"
JS_DIR        = SHOWCASE_DIR / "assets" / "js"
INDEX_PATH    = SHOWCASE_DIR / "index.html"

print(f"PROJECT_ROOT  = {PROJECT_ROOT}")
print(f"SHOWCASE_DIR  = {SHOWCASE_DIR.relative_to(PROJECT_ROOT)}")
print(f"TEMPLATE_PATH = {TEMPLATE_PATH.relative_to(PROJECT_ROOT)}")
print(f"SQL_DIR       = {SQL_DIR.relative_to(PROJECT_ROOT)}")
print(f"JS_DIR        = {JS_DIR.relative_to(PROJECT_ROOT)}")

## Hjelpefunksjoner

In [0]:
def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.strip() + "\n", encoding="utf-8")
    print(f"  Skrev: {path.relative_to(PROJECT_ROOT)}")


def write_json_as_js(path: Path, variable_name: str, payload: dict[str, Any]) -> None:
    content = (
        f"window.{variable_name} = "
        + json.dumps(payload, ensure_ascii=False, indent=2)
        + ";\n"
    )
    write_text(path, content)


def read_template(path: Path) -> Template:
    if not path.exists():
        raise FileNotFoundError(f"Fant ikke template-fil: {path}")
    return Template(path.read_text(encoding="utf-8"))


def read_query(name: str) -> str:
    path = SQL_DIR / f"{name}.sql"
    if not path.exists():
        raise FileNotFoundError(f"Fant ikke query-fil: {path}")
    return read_template(path).safe_substitute({
        "catalog": CATALOG,
        "schema": SCHEMA,
    })


def to_pandas(query: str) -> pd.DataFrame:
    return spark.sql(query).toPandas()


def normalize_percent_series(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce").fillna(0)
    if len(numeric) > 0 and numeric.max() <= 1.5:
        numeric = numeric * 100
    return numeric.round(1)


def format_int(value: Any) -> str:
    return f"{int(round(float(value))):,}".replace(",", " ")


def format_percent(value: Any, decimals: int = 2) -> str:
    return f"{float(value):.{decimals}f}".replace(".", ",")

## Valider kildefiler

In [0]:
required_source_files = [
    "showcase-page/templates/index.html.tpl",

    "showcase-page/sql/pensjonsandel_latest.sql",
    "showcase-page/sql/pensjonsandel_trend.sql",
    "showcase-page/sql/aldersfordeling.sql",
    "showcase-page/sql/top_kommuner.sql",
    "showcase-page/sql/top_naeringer.sql",
    "showcase-page/sql/aldersgruppe_trend.sql",
    "showcase-page/sql/kommuner_detalj.sql",

    "showcase-page/assets/css/main.css",
    "showcase-page/assets/css/tokens.css",
    "showcase-page/assets/css/base.css",
    "showcase-page/assets/css/layout.css",
    "showcase-page/assets/css/components.css",
    "showcase-page/assets/css/table.css",
    "showcase-page/assets/css/responsive.css",

    "showcase-page/assets/js/charts.js",
    "showcase-page/assets/js/table.js",
    "showcase-page/assets/js/main.js",
]

missing_source_files = [
    rel_path
    for rel_path in required_source_files
    if not (PROJECT_ROOT / rel_path).exists()
]

if missing_source_files:
    raise FileNotFoundError(
        "Mangler kildefiler i repoet:\n  " + "\n  ".join(missing_source_files)
    )

print(f"✓ Alle {len(required_source_files)} nødvendige kildefiler finnes")

## Les dashboarddata

Hver spørring leses fra en ekstern `.sql`-fil i `sql/github_dashboard/`.

Template-variabler `$catalog` og `$schema` substitueres automatisk.

### Leser data til showcase dashboardet som genereres.

In [0]:
# KPI-er: siste år
latest_df = to_pandas(read_query("pensjonsandel_latest"))

if latest_df.empty:
    raise ValueError("Fant ingen KPI-rad fra pensjonsandel_latest.sql")

latest = latest_df.iloc[0]

kpi_year           = int(latest["year"])
kpi_pensjonsandel  = float(latest["pensjonsandel_pst"])
kpi_55_pluss       = int(latest["total_55_pluss"])
kpi_total          = int(latest["total_befolkning"])

# Trend: pensjonsandel over tid
trend_df = to_pandas(read_query("pensjonsandel_trend"))

data_trend = {
    "years":  [int(v) for v in trend_df["year"].tolist()],
    "values": [float(v) for v in pd.to_numeric(trend_df["pensjonsandel_pst"]).round(1).tolist()],
}

# Aldersfordeling siste år
alder_df = to_pandas(read_query("aldersfordeling"))
alder_df["andel_pst"] = normalize_percent_series(alder_df["andel"])

data_alder = {
    "labels": alder_df["aldersgruppe"].astype(str).tolist(),
    "values": [int(v) for v in pd.to_numeric(alder_df["befolkning"]).astype(int).tolist()],
    "andel":  [float(v) for v in alder_df["andel_pst"].tolist()],
}

# Top kommuner
kommuner_df = to_pandas(read_query("top_kommuner"))

data_kommuner = {
    "labels": kommuner_df["kommune"].astype(str).tolist(),
    "values": [float(v) for v in pd.to_numeric(kommuner_df["andel_pst"]).round(1).tolist()],
}

# Top næringer
naering_df = to_pandas(read_query("top_naeringer"))

data_naering = {
    "labels": naering_df["naering"].astype(str).tolist(),
    "values": [float(v) for v in pd.to_numeric(naering_df["volum_mrd"]).round(2).tolist()],
}

# Aldersgruppe-trend
aldertrend_df = to_pandas(read_query("aldersgruppe_trend"))
aldertrend_df["andel_pst"] = normalize_percent_series(aldertrend_df["andel"])

years = sorted([int(v) for v in aldertrend_df["year"].astype(int).unique().tolist()])

groups_df = (
    aldertrend_df[["aldersgruppe", "aldersgruppe_sortering"]]
    .drop_duplicates()
    .sort_values("aldersgruppe_sortering")
)
groups = groups_df["aldersgruppe"].astype(str).tolist()

series = {}
for group in groups:
    group_df = aldertrend_df[aldertrend_df["aldersgruppe"] == group].copy()
    group_df = group_df.set_index("year").reindex(years)
    series[group] = [float(v) for v in group_df["andel_pst"].fillna(0).round(1).tolist()]

data_aldertrend = {
    "years":  years,
    "series": series,
}

# Kommune-detaljtabell
tabell_df = to_pandas(read_query("kommuner_detalj"))

data_tabell = [
    {
        "kommune":             str(row["kommune"]),
        "innbyggere":          int(row["innbyggere"]),
        "innbyggere_55_pluss": int(row["innbyggere_55_pluss"]),
        "andel_pst":           float(row["andel_pst"]),
    }
    for row in tabell_df.to_dict(orient="records")
]

print("✓ Data hentet")
print(f"  KPI-år:           {kpi_year}")
print(f"  Trendpunkter:     {len(data_trend['years'])}")
print(f"  Aldersgrupper:    {len(data_alder['labels'])}")
print(f"  Top kommuner:     {len(data_kommuner['labels'])}")
print(f"  Top næringer:     {len(data_naering['labels'])}")
print(f"  Kommuner i tabell: {len(data_tabell)}")

## Generer output-filer

### Generer `data.js`

Dashboard-payload skrives som `window.dashboardData = { ... }` til `assets/js/data.js`.

In [0]:
dashboard_payload = {
    "generatedDate": GENERATED_DATE,
    "dataTrend":     data_trend,
    "dataAlder":     data_alder,
    "dataKommuner":  data_kommuner,
    "dataNaering":   data_naering,
    "dataAlderTrend": data_aldertrend,
    "dataTabell":    data_tabell,
    "colors": [
        "#38BDF8", "#22C55E", "#F59E0B", "#A78BFA",
        "#F43F5E", "#14B8A6", "#F97316", "#EAB308",
    ],
}

write_json_as_js(JS_DIR / "data.js", "dashboardData", dashboard_payload)

### Generer `index.html` fra ekstern template

HTML genereres ved å lese `templates/index.html.tpl` og substituere KPI-plassholdere.

In [0]:
html = read_template(TEMPLATE_PATH).safe_substitute({
    "kpi_year":          kpi_year,
    "kpi_pensjonsandel": format_percent(kpi_pensjonsandel, decimals=2),
    "kpi_55_pluss":      format_int(kpi_55_pluss),
    "kpi_total":         format_int(kpi_total),
    "generated_date":    GENERATED_DATE,
})

write_text(INDEX_PATH, html)

print(f"✓ index.html generert ({len(html):,} tegn)")

## Sluttvalidering

In [ ]:
required_output_files = [
    "showcase-page/index.html",
    "showcase-page/assets/js/data.js",
]

In [ ]:
missing_output_files = [
    rel_path
    for rel_path in required_output_files
    if not (PROJECT_ROOT / rel_path).exists()
]

In [0]:
if missing_output_files:
    raise FileNotFoundError(
        "Mangler genererte filer: " + ", ".join(missing_output_files)
    )

print("✓ Ferdig")
print()
print("Genererte filer:")
for rel_path in required_output_files:
    print(f"  - {rel_path}")
print()
print("Neste steg:")
print("  1. Sjekk git diff")
print("  2. Commit index.html og assets/js/data.js hvis de er oppdatert")
print("  3. Push til main")